In [37]:
# Quoridor Game

In [50]:
from dataclasses import dataclass, replace
from collections import deque
import random
import math

import numpy as np
from sklearn.neural_network import MLPClassifier

In [39]:
N = 5 # board width and length
WN = N - 1 # wall slots form a WN x WN grid
WALLS_PER_PLAYER = 3

NORTH = (-1, 0)
SOUTH = (1, 0)
EAST = (0, 1)
WEST = (0, -1)
DIRECTIONS = (NORTH, SOUTH, EAST, WEST)

GOAL_ROW = (N - 1, 0) #player 0 aims for the bottom row, player 1 for the top

def slot_bit(r, c):
    # bit flat for wall slot (r, c)
    return 1 << (r * WN + c)

@dataclass(frozen=True)
class State:
    pawns: tuple #((r0, c0), (r1, c1)) positions for player 1 and 2
    walls_left: tuple # (walls_left_for_p0, walls_left_for_p1)
    h_walls: int # bitmask of occupied horizontal slots
    v_walls: int # bitmask of occupied vertical slots
    turn: int # 0 or 1, whose move it is

def initial_state():
    mid = N//2

    return State(
        pawns=((0, mid), (N - 1, mid)), # start the players in the middle column
        walls_left=(WALLS_PER_PLAYER, WALLS_PER_PLAYER),
        h_walls=0,
        v_walls=0,
        turn=0,
    )


In [40]:
def in_bounds(r, c):
    return 0 <=r < N and 0 <= c < N

def is_blocked(state, r, c, direction):
    dr, dc = direction
    if dr != 0:
        # step in the verical direction, the groove lies bewteen rows, so pick the slot row
        groove_row = r if dr == 1 else r - 1
        mask = state.h_walls
        slots = ((groove_row, c), (groove_row, c - 1))

    else:
        groove_col = c if dc == 1 else c - 1
        mask = state.v_walls
        slots = ((r, groove_col), (r - 1, groove_col))

    for sr, sc in slots:
        if 0 <=sr < WN and 0 <= sc < WN and mask & slot_bit(sr, sc):
            return True

    return False
        

In [41]:
s = initial_state()
s = State(s.pawns, s.walls_left, s.h_walls | slot_bit(1, 1), s.v_walls, s.turn)

print(is_blocked(s, 1, 1, SOUTH))   # True
print(is_blocked(s, 1, 2, SOUTH))   # True, same wall covers column 2
print(is_blocked(s, 1, 3, SOUTH))   # False
print(is_blocked(s, 2, 1, NORTH))   # True, blocking is symmetric
print(is_blocked(s, 1, 1, EAST))    # False, horizontal walls don't block sideways

True
True
False
True
False


In [42]:
def pawn_moves(state):
    # squares the player to move can step to, including jumps over the opponent
    me = state.pawns[state.turn]
    them = state.pawns[1-state.turn]
    moves = []

    for direction in DIRECTIONS:
        dr, dc = direction
        r, c = me[0] + dr, me[1] + dc

        if not in_bounds(r, c) or is_blocked(state, me[0], me[1], direction):
            continue
        if (r, c) != them:
            moves.append((r,c))
            continue

        # opponent is in the target square so we try jumping over
        jr, jc = r + dr, c + dc
        if in_bounds(jr, jc) and not is_blocked(state, r, c, direction):
            moves.append((jr, jc))
            continue

        # blocked behind them, so sidestep to either square perpendituclar to the player
        for side in ((dc, dr), (-dc, -dr)):
            sr, sc = r + side[0], c + side[1]
            if in_bounds(sr, sc) and not is_blocked(state, r, c, side):
                moves.append((sr, sc))

    return moves

In [43]:
def path_exists(state, player):
    # BFS can this player still reach their goal row?
    goal = GOAL_ROW[player]
    start = state.pawns[player]
    seen = {start}
    q = deque([start])

    while q:
        r, c = q.popleft()
        if r == goal:
            return True
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            if (in_bounds(nr, nc) 
            and (nr, nc) not in seen 
            and not is_blocked(state, r, c, (dr, dc))
            ):
                seen.add((nr, nc))
                q.append((nr, nc))

    return False

In [44]:
def h_conflict(state, r, c):
    """
    a horizontal wall at (r, c) covers columns c and c+1 so a horizontal
    neighbour one column either side would overlap half of it
    """
    for cc in (c - 1, c + 1):
        if 0 <= cc < WN and state.h_walls & slot_bit(r, cc):
            return True

    return False

def v_conflict(state, r, c):
    for rr in (r - 1, r + 1):
        if 0 <=rr < WN and state.v_walls & slot_bit(rr, c):
            return True
    return False

def wall_moves(state):
    if state.walls_left[state.turn] == 0:
        return []

    moves = []
    for r in range(WN):
        for c in range(WN):
            bit = slot_bit(r, c)


            # nothing of either oreintation may already sit on this intersection
            if (state.h_walls | state.v_walls) & bit:
                continue
                
            if not h_conflict(state, r, c):
                cand = replace(state, h_walls=state.h_walls | bit)
                if path_exists(cand, 0) and path_exists(cand, 1):
                    moves.append(('H', r, c))

            if not v_conflict(state, r, c):
                cand = replace(state, v_walls=state.v_walls | bit)
                if path_exists(cand, 0) and path_exists(cand, 1):
                    moves.append(('V', r, c))

    return moves

In [45]:
def legal_moves(state):
    return [('P', r, c) for r, c in pawn_moves(state)] + wall_moves(state)

def apply_move(state, move):
    # the successor function, every agent's compute cost is counted in calls to this
    kind, r, c = move
    nxt = 1 - state.turn

    if kind == 'P':
        pawns = list(state.pawns)
        pawns[state.turn] = (r, c)
        return replace(state, pawns=tuple(pawns), turn=nxt)

    walls_left = list(state.walls_left)
    walls_left[state.turn] -= 1
    bit = slot_bit(r, c)

    if kind == 'H':
        return replace(state, walls_left=tuple(walls_left), h_walls=state.h_walls | bit, turn=nxt)
    return replace(state, walls_left=tuple(walls_left), v_walls=state.v_walls | bit, turn=nxt)

def winner(state):
    # 0 or 1 if that player reached their goal row, otherwise None
    for player in (0, 1):
        if state.pawns[player][0] == GOAL_ROW[player]:
            return player
    return None

In [48]:
def random_move(state, rng):
    # one uniformly random legal move, without validating the walls we dont play
    candidates = [('P', r, c) for r, c in pawn_moves(state)]

    if state.walls_left[state.turn] > 0:
        for r in range(WN):
            for c in range(WN):
                bit = slot_bit(r, c)
                if (state.h_walls | state.v_walls) & bit:
                    continue
                if not h_conflict(state, r, c):
                    candidates.append(('H', r, c))
                if not v_conflict(state, r, c):
                    candidates.append(('V', r, c))

    while candidates:
        move = rng.choice(candidates)
        if move[0] == 'P':
            return move
        cand = apply_move(state, move)
        if path_exists(cand, 0) and path_exists(cand, 1):
            return move
        candidates.remove(move) # that wall cuts someone off, never offer it again

    return None

In [49]:
def random_playout(rng, max_plies=300):
    state = initial_state()
    for _ in range(max_plies):
        w = winner(state)
        if w is not None:
            return w

        state = apply_move(state, random_move(state, rng))
    return None

# s = initial_state()
# rng = random.Random(0)
# res = [random_playout(rng) for _ in range(1000)]

# print("opening moves:", len(legal_moves(s)), "=", len(pawn_moves(s)), "pawn +", len(wall_moves(s)), "wall")
# print(f"P0 {res.count(0)}  P1 {res.count(1)}  unfinished {res.count(None)}")

opening moves: 35 = 3 pawn + 32 wall
P0 444  P1 536  unfinished 20


In [64]:
def rollout(state, rng, max_plies=300):
    for _ in range(max_plies):
        w = winner(state)
        if w is not None:
            return w
        state = apply_move(state, random_move(state, rng))
    return None

class Node:
    def __init__(self, state, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move
        self.children = []
        # finished game has no moves to try, wihch is what stops the search descending past a win
        self.untried = [] if winner(state) is not None else legal_moves(state)
        self.visits = 0
        self.wins = 0.0

def uct_score(child, parent_visits, c=1.4):
    return child.wins / child.visits + c * math.sqrt(math.log(parent_visits) / child.visits)

def mcts_root(state, rng, simulations=200, c=1.4):
    root = Node(state)

    for _ in range(simulations):
        node = root

        # select: descend while this node is fully expanded and has somewhere to go
        while not node.untried and node.children:
            node = max(node.children, key=lambda ch: uct_score(ch, node.visits, c))

        # expand
        if node.untried:
            move = node.untried.pop(rng.randrange(len(node.untried)))
            child = Node(apply_move(node.state, move), parent=node, move=move)
            node.children.append(child)
            node = child

        # simulate
        result = rollout(node.state, rng)

        # backpropagate
        while node is not None:
            node.visits += 1
            if result is None:
                node.wins += 0.5
            elif result == 1 - node.state.turn:
                node.wins += 1.0
            node = node.parent

    return root

In [65]:
def encode(state):
    # flat features: each pawn one-hot over the board, both wall masks as bits,
    # walls remaining, and whose turn it is
    v = [0.0] * (2 * N * N + 2 * WN * WN + 3)
    i = 0

    for p in (0, 1):
        r, c = state.pawns[p]
        v[i + r * N + c] = 1.0
        i += N * N

    for mask in (state.h_walls, state.v_walls):
        for s in range(WN * WN):
            if mask >> s & 1:
                v[i + s] = 1.0
        i += WN * WN

    v[i] = state.walls_left[0] / WALLS_PER_PLAYER
    v[i + 1] = state.walls_left[1] / WALLS_PER_PLAYER
    v[i + 2] = float(state.turn)
    return v

In [66]:
# every move the game can ever contain gets a fixed index, so the policy output
# is a vector of constant length and illegal moves are masked out at use time
ALL_MOVES = ([('P', r, c) for r in range(N) for c in range(N)]
             + [(k, r, c) for k in ('H', 'V') for r in range(WN) for c in range(WN)])
MOVE_INDEX = {m: i for i, m in enumerate(ALL_MOVES)}
POLICY_SIZE = len(ALL_MOVES)   # 25 pawn targets + 32 wall slots = 57

In [69]:
def mcts_move(state, rng, simulations=200, c=1.4):
    root = mcts_root(state, rng, simulations, c)
    return max(root.children, key=lambda ch: ch.visits).move

def visits_to_label(root, temperature=0.25):
    weights = [ch.visits ** (1.0 / temperature) for ch in root.children]
    total = sum(weights)
    label = [0.0] * POLICY_SIZE
    for ch, w in zip(root.children, weights):
        label[MOVE_INDEX[ch.move]] = w / total
    return label

def mcts_policy(state, rng, simulations=200, c=1.4, temperature=0.25):
    return visits_to_label(mcts_root(state, rng, simulations, c), temperature)

def generate_data(games, rng, simulations=200, c=1.4, temperature=0.25, max_plies=300):
    X, Y = [], []
    for _ in range(games):
        state = initial_state()
        for _ in range(max_plies):
            if winner(state) is not None:
                break
            root = mcts_root(state, rng, simulations, c)

            X.append(encode(state))
            Y.append(visits_to_label(root, temperature))

            # the stored label is sharpened, but the game advances by sampling raw
            # visit counts, so you collect varied positions instead of one line replayed
            moves = [ch.move for ch in root.children]
            visits = [ch.visits for ch in root.children]
            state = apply_move(state, rng.choices(moves, weights=visits)[0])

    return X, Y

In [70]:
def train_policy(X, Y, model):
    # the search's distribution gets collapsed to its single best move, since
    # sklearn classifiers take a class label rather than a target distribution
    Xa = np.asarray(X, dtype=np.float32)
    ya = np.asarray([int(np.argmax(y)) for y in Y])
    model.fit(Xa, ya)
    return model

def model_agent(model):
    # plays by scoring all 57 slots and taking the best legal one
    def choose(state, rng):
        probs = model.predict_proba(np.asarray([encode(state)], dtype=np.float32))[0]
        scores = dict(zip(model.classes_, probs))
        return max(legal_moves(state), key=lambda m: scores.get(MOVE_INDEX[m], 0.0))
    return choose

In [71]:
def flip_mask(mask):
    out = 0
    for r in range(WN):
        for c in range(WN):
            if mask >> (r * WN + c) & 1:
                out |= slot_bit(WN - 1 - r, c)
    return out

def canonical_state(state):
    # present the board so the player to move always travels toward higher rows,
    # so one model covers both sides instead of learning two mirrored policies
    if state.turn == 0:
        return state
    (r0, c0), (r1, c1) = state.pawns
    return State(
        pawns=((N - 1 - r1, c1), (N - 1 - r0, c0)),
        walls_left=(state.walls_left[1], state.walls_left[0]),
        h_walls=flip_mask(state.h_walls),
        v_walls=flip_mask(state.v_walls),
        turn=0,
    )

def canonical_move(move):
    # its own inverse, so one call maps a move into the canonical frame or back out
    kind, r, c = move
    return (kind, (N - 1 - r) if kind == 'P' else (WN - 1 - r), c)